In [2]:
import polars as pl

In [4]:
(
    pl
    .read_parquet(
        '/Users/lowell/Projects/alt_nfp/output/precision_budget.parquet'
    )
)

source,n_obs,precision_per_obs,total_precision,lambda_mean,alpha_mean,share
str,i64,f64,f64,f64,f64,f64
"""CES SA""",158,105968.798165,1.6743e7,0.966977,-0.000358,0.06447
"""CES NSA""",158,188975.623107,2.9858e7,0.966977,-0.000358,0.11497
"""G""",83,745.620861,61886.531458,1.024834,0.008709,0.000238
"""QCEW (M2)""",54,3.2835e6,1.7731e8,1.0,0.0,0.682733
"""QCEW (M3+M1)""",107,333950.756688,3.5733e7,1.0,0.0,0.13759


In [ ]:
from datetime import date
import numpy as np

# CES SA employment levels (thousands) from the full uncensored panel
prev_level_k = [
    157304, 157032, 157238, 157466, 157530, 157608, 157695, 157748,
    157757, 157912, 157945, 158079, 158942, 158268, 158310, 158377,
    158485, 158498, 158478, 158542, 158472, 158548, 158408, 158449,
    158497,
]

backtest_results = pl.DataFrame({
    "ref_date": [
        date(2024, 1, 12), date(2024, 2, 12), date(2024, 3, 12), date(2024, 4, 12),
        date(2024, 5, 12), date(2024, 6, 12), date(2024, 7, 12), date(2024, 8, 12),
        date(2024, 9, 12), date(2024, 10, 12), date(2024, 11, 12), date(2024, 12, 12),
        date(2025, 1, 12), date(2025, 2, 12), date(2025, 3, 12), date(2025, 4, 12),
        date(2025, 5, 12), date(2025, 6, 12), date(2025, 7, 12), date(2025, 8, 12),
        date(2025, 9, 12), date(2025, 10, 12), date(2025, 11, 12), date(2025, 12, 12),
        date(2026, 1, 12),
    ],
    "actual_growth_pct": [
        0.447, 0.150, 0.196, 0.068, 0.136, 0.074, 0.091, 0.049,
        0.160, 0.027, -0.283, 0.203, 0.070, 0.064, 0.075, 0.099,
        0.012, -0.008, 0.045, -0.016, 0.068, -0.745, 0.026, -0.016,
        -0.565,
    ],
    "nowcast_growth_pct": [
        0.308, 0.131, 0.336, 0.026, 0.183, 0.032, 0.146, 0.140,
        0.030, 0.070, 0.124, 0.029, 0.360, -0.409, 0.325, -0.064,
        0.177, -0.110, 0.181, -0.046, -0.006, 0.306, 0.331, 0.216,
        0.221,
    ],
    "prev_level_k": prev_level_k,
}).with_columns(
    (pl.col("prev_level_k") * ((pl.col("actual_growth_pct") / 100).exp() - 1))
    .round(0)
    .cast(pl.Int64)
    .alias("actual_jobs_added_k"),
    (pl.col("prev_level_k") * ((pl.col("nowcast_growth_pct") / 100).exp() - 1))
    .round(0)
    .cast(pl.Int64)
    .alias("nowcast_jobs_added_k"),
).with_columns(
    (pl.col("actual_jobs_added_k") - pl.col("nowcast_jobs_added_k"))
    .alias("error_jobs_k"),
)

backtest_results